In [ ]:
# Prompts
""" I already have a PPO model using FinRL to suggest what stocks to buy at what price and what date
I need to implement something for A2C model as well
What would be the best use cases or ways in which A2C can be implemented? """

""" A2C for Asset Selection Instead of Direct Trading seems very interesting
If I am implementing this, would this model run once before the PPO model or would it run for every single timestep of the PPO model """

""" Please implement this
I have a list of stocks from toronto stock exchange in a csv file. I downloaded this data from yahoo finance API so the columns are same """

In [1]:
!pip install stable-baselines3

In [34]:
import numpy as np
import pandas as pd
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import A2C, PPO

class StockTradingEnv(gym.Env):
    def __init__(self, df):
        super(StockTradingEnv, self).__init__()
        self.df = df
        self.current_step = 0
        self.initial_balance = 10000
        self.balance = self.initial_balance
        self.shares_held = 0
        self.total_shares = len(df['tic'].unique())

        self.action_space = spaces.Discrete(3)  # Buy, Sell, Hold
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(len(df.columns)-2,), dtype=np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed, options=options)
        self.current_step = 0
        self.balance = self.initial_balance
        self.shares_held = 0
        obs = self._next_observation()
        return obs, {}  # Return observation and empty info dictionary

    def _next_observation(self):
        obs = self.df.iloc[self.current_step].drop(['date', 'tic']).values.astype(np.float32)
        return obs

    def step(self, action):
      current_price = self.df.iloc[self.current_step]['close']

      if action == 0:  # Buy
          num_shares = self.balance // current_price
          self.shares_held += num_shares
          self.balance -= num_shares * current_price
      elif action == 1:  # Sell
          self.balance += self.shares_held * current_price
          self.shares_held = 0

      self.current_step += 1

      # Ensure termination after a set number of steps per episode
      max_steps_per_stock = 200  # Adjust this based on data length
      done = self.current_step >= min(len(self.df) - 1, max_steps_per_stock)

      reward = self.balance + (self.shares_held * current_price) - self.initial_balance
      print(f"Step: {self.current_step}, Done: {done}, Balance: {self.balance}")

      return self._next_observation(), reward, done, False, {}  # False for truncated


In [4]:
df = pd.read_excel("TSX Chart Data.xlsx")

In [35]:
env = StockTradingEnv(df)

a2c_env = StockTradingEnv(df)
a2c_model = A2C("MlpPolicy", a2c_env, verbose=1)
a2c_model.learn(total_timesteps=10000)

Streaming output truncated to the last 5000 lines.
Step: 171, Done: False, Balance: 78909.42049005628
Step: 172, Done: False, Balance: 78909.42049005628
Step: 173, Done: False, Balance: 0.2892754375934601
Step: 174, Done: False, Balance: 78909.42049005628
Step: 175, Done: False, Balance: 78909.42049005628
Step: 176, Done: False, Balance: 78909.42049005628
Step: 177, Done: False, Balance: 0.2792843282222748
Step: 178, Done: False, Balance: 0.2792843282222748
Step: 179, Done: False, Balance: 0.2792843282222748
Step: 180, Done: False, Balance: 0.2792843282222748
Step: 181, Done: False, Balance: 0.2792843282222748
Step: 182, Done: False, Balance: 0.2792843282222748
Step: 183, Done: False, Balance: 0.2792843282222748
Step: 184, Done: False, Balance: 0.2792843282222748
Step: 185, Done: False, Balance: 0.2792843282222748
Step: 186, Done: False, Balance: 102925.25714221597
Step: 187, Done: False, Balance: 102925.25714221597
Step: 188, Done: False, Balance: 102925.25714221597
Step: 189, Done: F

In [36]:
def select_top_stocks(df, model, top_n=10, max_steps=200):
    stock_scores = {}

    for tic in df['tic'].unique():
        stock_df = df[df['tic'] == tic].reset_index(drop=True)  # Ensure isolated stock data

        temp_env = StockTradingEnv(stock_df)  # Use a separate environment for each stock
        obs, _ = temp_env.reset()
        total_reward = 0
        done = False
        step_count = 0  # Step counter

        while not done and step_count < max_steps:
            action, _ = model.predict(obs)
            obs, reward, done, _, _ = temp_env.step(action)
            total_reward += reward
            step_count += 1  # Increment step counter

        if step_count >= max_steps:
            print(f"Warning: Stock {tic} reached max_steps ({max_steps}) before termination.")

        stock_scores[tic] = total_reward

    return sorted(stock_scores, key=stock_scores.get, reverse=True)[:top_n]

In [32]:
# Step 2: Select top N stocks
top_stocks = select_top_stocks(df, a2c_model, top_n=10)
print("Selected Stocks:", top_stocks)

KeyboardInterrupt: 